In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from google.colab import files

# 1. Upload your Delhi weather + AQI CSV file
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload your Delhi weather + AQI CSV file.")

CSV = next(iter(uploaded))
OUT = Path("charts")
OUT.mkdir(parents=True, exist_ok=True)

print("CSV file loaded:", CSV)

# 2. Read and validate the data
df = pd.read_csv(CSV)

required_columns = [
    "date_ist", "time_ist", "location",
    "pm2_5", "pm10", "no2", "co", "aqi_index",
    "temp_c", "humidity", "pressure_mb", "windspeed_kph",
    "condition_text"
]
missing = [column for column in required_columns if column not in df.columns]
if missing:
    raise ValueError(
        "These required columns are missing from your CSV: "
        + ", ".join(missing)
        + "\nPlease check that you uploaded the correct Delhi weather + AQI dataset."
    )

# Convert numeric columns safely
numeric_columns = [
    "pm2_5", "pm10", "no2", "co", "aqi_index",
    "temp_c", "humidity", "pressure_mb", "windspeed_kph"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["date_ist"] = df["date_ist"].astype(str).str.strip()
df["time_ist"] = df["time_ist"].astype(str).str.strip()
df["dt"] = pd.to_datetime(
    df["date_ist"] + " " + df["time_ist"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

df = df.dropna(subset=["dt"]).copy()
if df.empty:
    raise ValueError(
        "No valid date/time rows were found. Expected date_ist format DD/MM/YYYY "
        "and time_ist format HH:MM."
    )

df["month"] = df["dt"].dt.month
df["hour"] = df["dt"].dt.hour
df["weekday"] = df["dt"].dt.dayofweek
df["date"] = df["dt"].dt.normalize()

# Remove rows with no station name
df["location"] = df["location"].astype(str).str.strip()
df = df[df["location"].ne("") & df["location"].ne("nan")].copy()

# 3. Chart styling
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100",
    "#e87ba4", "#008300", "#6250d6", "#e34948"
)
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
DIVERGING = LinearSegmentedColormap.from_list("div", [BLUE, "#e9e8e4", ORANGE])
SEQ = "YlOrRd"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.titlelocation": "left",
    "font.size": 10,
    "legend.frameon": False,
    "figure.dpi": 110
})

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
LIMIT_PM25, LIMIT_PM10 = 60, 100  # 24-hour standards used in the original analysis

POLL = ["pm2_5", "pm10", "no2", "co", "aqi_index"]
WEATHER = ["temp_c", "humidity", "pressure_mb", "windspeed_kph"]
LABEL = {
    "pm2_5": "PM2.5 (µg/m³)",
    "pm10": "PM10 (µg/m³)",
    "no2": "NO₂ (µg/m³)",
    "co": "CO (µg/m³)",
    "aqi_index": "AQI index",
    "temp_c": "Temperature (°C)",
    "humidity": "Humidity (%)",
    "pressure_mb": "Pressure (mb)",
    "windspeed_kph": "Wind speed (km/h)"
}
ALL = POLL + WEATHER

# Check that the required numeric fields contain usable values
usable = [c for c in ALL if df[c].notna().any()]
if len(usable) != len(ALL):
    empty_cols = sorted(set(ALL) - set(usable))
    raise ValueError("These numeric columns contain no valid numbers: " + ", ".join(empty_cols))

# Station groups: locations with matching PM10 readings over the first 200 timestamps
sig = df.pivot_table(index="dt", columns="location", values="pm10")
groups = {}
for loc in sig.columns:
    key = tuple(np.round(sig[loc].values[:200], 1))
    groups.setdefault(key, []).append(loc)

ZONES = {}
for i, locs in enumerate(groups.values()):
    zone_name = f"Zone {i + 1}"
    for loc in locs:
        ZONES[loc] = zone_name

df["zone"] = df["location"].map(ZONES).fillna("Unknown")

print("\nStation zones (locations with matching PM10 series):")
for z in sorted(set(ZONES.values())):
    print(" ", z, [loc for loc, zone in ZONES.items() if zone == z])

city = df.groupby("dt")[ALL].mean(numeric_only=True)
daily = df.groupby("date")[ALL].mean(numeric_only=True)

# Save each chart AND display it directly in Google Colab
def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Displayed and saved:", name)

# 01 - Distributions of all parameters
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, c in zip(axes.flat, ALL):
    x = df[c].dropna()
    hi = x.quantile(0.995) if c in ("aqi_index", "pm10", "co") else x.max()
    shown = x[x <= hi]
    ax.hist(shown, bins=50, color=BLUE, edgecolor=SURFACE, linewidth=0.4)
    ax.axvline(x.median(), color=ORANGE, lw=2)
    ax.set_title(LABEL[c])
    ax.set_ylabel("hours")
    ax.text(
        0.97, 0.93,
        f"median {x.median():.1f}\nmean {x.mean():.1f}",
        transform=ax.transAxes, ha="right", va="top",
        color=MUTED, fontsize=9
    )
fig.suptitle(
    "Distribution of every parameter (orange = median; selected skewed series trimmed at P99.5)",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "01_distributions.png")

# 02 - Daily time series
fig, axes = plt.subplots(len(ALL), 1, figsize=(13, 17), sharex=True)
for ax, c in zip(axes, ALL):
    ax.plot(daily.index, daily[c], color=BLUE, lw=0.9, alpha=0.55)
    ax.plot(
        daily.index,
        daily[c].rolling(7, center=True).mean(),
        color=ORANGE, lw=2
    )
    ax.set_ylabel(LABEL[c], fontsize=9)
    if c == "pm2_5":
        ax.axhline(LIMIT_PM25, color=MUTED, ls="--", lw=1)
    if c == "pm10":
        ax.axhline(LIMIT_PM10, color=MUTED, ls="--", lw=1)
axes[0].set_title(
    "Daily city-wide mean through the dataset (blue = daily, orange = 7-day average)"
)
save(fig, "02_daily_timeseries.png")

# 03 - Monthly averages
mon = df.groupby("month")[ALL].mean(numeric_only=True).reindex(range(1, 13))
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, c in zip(axes.flat, ALL):
    ax.bar(MONTHS, mon[c], color=BLUE, width=0.7)
    ax.set_title(LABEL[c])
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.axhline(mon[c].mean(), color=ORANGE, lw=1.5)
fig.suptitle(
    "Monthly average of every parameter (orange = annual mean)",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "03_monthly_averages.png")

# 04 - Hour-of-day profile
hr = df.groupby("hour")[ALL].mean(numeric_only=True).reindex(range(24))
fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
for ax, c in zip(axes.flat, ALL):
    ax.plot(hr.index, hr[c], color=BLUE, lw=2.2)
    ax.fill_between(hr.index, hr[c], hr[c].min(), color=BLUE, alpha=0.12)
    ax.set_title(LABEL[c])
    ax.set_xticks(range(0, 24, 6))
fig.suptitle(
    "Average daily cycle (hour of day, IST)",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "04_hourly_profile.png")

# 05 - Hour x month heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, c in zip(axes, ["pm2_5", "pm10"]):
    pv = df.pivot_table(
        values=c, index="hour", columns="month", aggfunc="mean"
    ).reindex(index=range(24), columns=range(1, 13))
    im = ax.imshow(pv, aspect="auto", cmap=SEQ, origin="lower")
    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTHS, fontsize=8)
    ax.set_yticks(range(0, 24, 3))
    ax.set_ylabel("hour of day")
    ax.grid(False)
    ax.set_title(f"{LABEL[c]} by hour and month")
    fig.colorbar(im, ax=ax, shrink=0.85)
save(fig, "05_hour_month_heatmap.png")

# 06 - Calendar heatmap of daily PM2.5
cal = daily["pm2_5"].to_frame("v")
cal["m"] = cal.index.month
cal["d"] = cal.index.day
pv = cal.pivot(index="m", columns="d", values="v").reindex(
    index=range(1, 13), columns=range(1, 32)
)
fig, ax = plt.subplots(figsize=(14, 4.8))
im = ax.imshow(pv, aspect="auto", cmap=SEQ)
ax.set_yticks(range(12))
ax.set_yticklabels(MONTHS)
ax.set_xticks(range(0, 31, 5))
ax.set_xticklabels(range(1, 32, 5))
ax.grid(False)
ax.set_title("Daily mean PM2.5 calendar (µg/m³): darker = higher value")
ax.set_xlabel("day of month")
fig.colorbar(im, ax=ax, shrink=0.9)
save(fig, "06_pm25_calendar.png")

# 07 - Station zones: pollutant boxplots and mean CO by station
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
zone_names = sorted(df["zone"].dropna().unique())
palette = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]
zone_cols = {z: palette[i % len(palette)] for i, z in enumerate(zone_names)}

for ax, c in zip(axes.flat[:5], POLL):
    data = [
        df.loc[df.zone == z, c].dropna().clip(upper=df[c].quantile(0.99))
        for z in zone_names
    ]
    if data and any(len(values) for values in data):
        bp = ax.boxplot(
            data, labels=zone_names, patch_artist=True, widths=0.5,
            medianprops=dict(color=INK, lw=2),
            flierprops=dict(markersize=2, alpha=0.3)
        )
        for patch, z in zip(bp["boxes"], zone_names):
            patch.set_facecolor(zone_cols[z])
            patch.set_alpha(0.75)
    ax.set_title(LABEL[c])
    ax.tick_params(axis="x", rotation=30, labelsize=8)

st = df.groupby("location")[["co"]].mean().dropna()
ax = axes.flat[5]
if not st.empty:
    order = st.sort_values("co").index
    ax.barh(
        order, st.loc[order, "co"],
        color=[zone_cols.get(ZONES.get(loc, "Unknown"), BLUE) for loc in order]
    )
ax.set_title("Mean CO by station")
fig.suptitle(
    "Station zones: " + " | ".join(
        f"{z}: {', '.join(loc for loc, v in ZONES.items() if v == z)}"
        for z in zone_names
    ),
    x=0.01, ha="left", fontsize=9, color=MUTED
)
save(fig, "07_station_zones.png")

# 08 - Correlation heatmap
corr = city[ALL].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap=DIVERGING, vmin=-1, vmax=1)
ax.grid(False)
names = [LABEL[c].split(" (")[0] for c in ALL]
ax.set_xticks(range(len(ALL)))
ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(len(ALL)))
ax.set_yticklabels(names)
for i in range(len(ALL)):
    for j in range(len(ALL)):
        value = corr.iloc[i, j]
        if pd.notna(value):
            ax.text(
                j, i, f"{value:.2f}", ha="center", va="center",
                fontsize=8,
                color=INK if abs(value) < 0.6 else "white"
            )
ax.set_title("Correlation matrix (hourly city-wide means)")
fig.colorbar(im, ax=ax, shrink=0.8)
save(fig, "08_correlation_heatmap.png")

# 09 - Weather vs PM2.5
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, c in zip(axes.flat, WEATHER):
    valid = df[[c, "pm2_5"]].dropna()
    if len(valid) >= 10 and valid[c].nunique() > 1:
        bins = pd.qcut(valid[c], 10, duplicates="drop")
        grouped = valid.groupby(bins, observed=True)["pm2_5"]
        means = grouped.mean()
        q25 = grouped.quantile(0.25)
        q75 = grouped.quantile(0.75)
        mid = np.array([interval.mid for interval in means.index])
        ax.fill_between(mid, q25.to_numpy(), q75.to_numpy(), color=BLUE, alpha=0.2)
        ax.plot(mid, means.to_numpy(), color=BLUE, lw=2.2, marker="o", ms=5)
    ax.set_xlabel(LABEL[c])
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_title(f"PM2.5 vs {LABEL[c].split(' (')[0].lower()}")
fig.suptitle(
    "How weather relates to PM2.5 (line = mean per decile, band = interquartile range)",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "09_weather_vs_pm25.png")

# 10 - Hexbin scatter relationships
pairs = [
    ("pm2_5", "pm10"), ("pm2_5", "co"), ("co", "no2"),
    ("temp_c", "humidity"), ("temp_c", "pressure_mb"),
    ("windspeed_kph", "no2")
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8.5))
sub = df.sample(n=min(20000, len(df)), random_state=1)
for ax, (a, b) in zip(axes.flat, pairs):
    valid = sub[[a, b]].dropna()
    if not valid.empty:
        hb = ax.hexbin(
            valid[a], valid[b], gridsize=35,
            cmap="Blues", mincnt=1, bins="log"
        )
    r = df[[a, b]].corr().iloc[0, 1]
    ax.set_xlabel(LABEL[a])
    ax.set_ylabel(LABEL[b])
    ax.set_title(f"r = {r:.2f}" if pd.notna(r) else "r = unavailable")
fig.suptitle(
    "Relationships between parameters (hexbin density, up to 20k-row sample)",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "10_scatter_relationships.png")

# 11 - Sky condition vs pollutants
cond = (
    df.groupby("condition_text")[["pm2_5", "pm10", "no2"]]
    .mean(numeric_only=True).sort_values("pm2_5")
)
cnt = df["condition_text"].value_counts()
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)
for ax, c in zip(axes, ["pm2_5", "pm10", "no2"]):
    ax.barh(cond.index.astype(str), cond[c], color=BLUE)
    ax.set_title(f"Mean {LABEL[c]}")
axes[0].set_yticks(range(len(cond)))
axes[0].set_yticklabels([
    f"{condition} (n={cnt.get(condition, 0):,})" for condition in cond.index
])
fig.suptitle(
    "Average pollution by sky condition",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "11_condition_vs_pollution.png")

# 12 - Days above selected PM limits per month and zone
dz = (
    df.groupby(["zone", "date"])[["pm2_5", "pm10"]]
    .mean(numeric_only=True).reset_index()
)
dz["month"] = dz["date"].dt.month
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
w = 0.38
for ax, c, lim in zip(axes, ["pm2_5", "pm10"], [LIMIT_PM25, LIMIT_PM10]):
    for k, (z, col) in enumerate(zone_cols.items()):
        d = dz[dz.zone == z]
        n = (
            (d[c] > lim).groupby(d.month).sum()
            .reindex(range(1, 13), fill_value=0)
        )
        offset = (k - (len(zone_cols) - 1) / 2) * w
        ax.bar(np.arange(12) + offset, n.values, w * 0.92, color=col, label=z)
    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTHS, fontsize=8)
    ax.set_ylabel("days above selected limit")
    if zone_cols:
        ax.legend()
    ax.set_title(f"{c.upper().replace('_', '.')} > {lim} µg/m³")
save(fig, "12_limit_exceedance.png")

# 13 - AQI category mix per month
bands = [0, 50, 100, 200, 300, 400, 500, np.inf]
band_names = [
    "Good", "Satisfactory", "Moderate", "Poor",
    "Very poor", "Severe", "Beyond 500"
]
band_colors = [
    "#1baf7a", "#8fc400", "#eda100", "#eb6834",
    "#e34948", "#8a1f1f", "#2b2b2b"
]
df["band"] = pd.cut(df["aqi_index"], bands, labels=band_names)
mix = (
    pd.crosstab(df["month"], df["band"], normalize="index") * 100
).reindex(index=range(1, 13), columns=band_names, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5.5))
bottom = np.zeros(12)
for band_name, col in zip(band_names, band_colors):
    values = mix[band_name].to_numpy()
    ax.bar(
        MONTHS, values, bottom=bottom, color=col,
        label=band_name, width=0.75,
        edgecolor=SURFACE, linewidth=1.2
    )
    bottom += values
ax.set_ylabel("% of hours")
ax.set_title("Share of hours in each AQI band, by month")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.08), fontsize=8)
save(fig, "13_aqi_band_mix.png")

# 14 - Weekday profile and 10 worst PM2.5 days
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
wd = df.groupby("weekday")[["pm2_5", "pm10", "no2"]].mean(numeric_only=True)
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
wd = wd.reindex(range(7))
axes[0].bar(days, wd["pm2_5"], color=BLUE)
axes[0].set_title("PM2.5 by weekday")
axes[1].bar(days, wd["no2"], color=ORANGE)
axes[1].set_title("NO₂ by weekday")
top = daily["pm2_5"].nlargest(10).sort_values()
axes[2].barh(
    [d.strftime("%d %b") for d in top.index],
    top.values, color=RED
)
axes[2].set_title("10 worst days by mean PM2.5")
save(fig, "14_weekday_and_worst_days.png")

# 15 - Seasonal boxplots
season = df["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Summer", 4: "Summer", 5: "Summer",
    6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
    10: "Post-monsoon", 11: "Post-monsoon"
})
season_order = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
season_columns = ["pm2_5", "pm10", "no2", "temp_c", "humidity", "windspeed_kph"]
for ax, c in zip(axes.flat, season_columns):
    data = [
        df.loc[season == s, c].dropna().clip(upper=df[c].quantile(0.99))
        for s in season_order
    ]
    bp = ax.boxplot(
        data, labels=season_order, patch_artist=True, widths=0.5,
        medianprops=dict(color=INK, lw=2),
        flierprops=dict(markersize=2, alpha=0.3)
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(BLUE)
        patch.set_alpha(0.7)
    ax.set_title(LABEL[c])
    ax.tick_params(axis="x", labelsize=8, rotation=20)
fig.suptitle(
    "Seasonal comparison (Winter = Dec-Feb, Summer = Mar-May, "
    "Monsoon = Jun-Sep, Post-monsoon = Oct-Nov)",
    x=0.01, ha="left", fontweight="bold"
)
save(fig, "15_seasonal_boxplots.png")

# 4. Numeric summary and key results
summary = df[ALL].describe(percentiles=[0.05, 0.5, 0.95]).T.round(2)
summary.to_csv(OUT / "key_parameters_summary.csv")

print("\nKEY PARAMETERS\n")
display(summary)

dz_summary = df.groupby(["zone", "date"])[["pm2_5", "pm10"]].mean()
print("\nDays above PM2.5 selected limit:", {
    z: int((dz_summary.loc[z, "pm2_5"] > LIMIT_PM25).sum())
    for z in dz_summary.index.get_level_values(0).unique()
})
print("Days above PM10 selected limit:", {
    z: int((dz_summary.loc[z, "pm10"] > LIMIT_PM10).sum())
    for z in dz_summary.index.get_level_values(0).unique()
})
print("Worst PM2.5 day:", daily["pm2_5"].idxmax().date(),
      round(daily["pm2_5"].max(), 1))
print("Best PM2.5 day:", daily["pm2_5"].idxmin().date(),
      round(daily["pm2_5"].min(), 1))

# 5. Download all PNG charts and the summary CSV as one ZIP file
import shutil
zip_path = shutil.make_archive("delhi_weather_aqi_charts", "zip", OUT)
print("\nAll charts and the summary CSV are saved in:", zip_path)
files.download(zip_path)
